In [12]:
import pandas as pd

In [13]:
for _ in ["legend_area"]:
    inpt = f"data/{_}.csv" 
    out = f"data/{_}_clean.csv" 

    df = pd.read_csv(inpt)
    # df.head()

    df["img"] = df["img1"].combine_first(df["img2"])
    df.drop(columns=['img1', 'img2'], inplace=True)


    df.to_csv(out, index=False)

In [14]:
pd.read_csv("data/legend_area.csv").head()

,Symbol,img1,img2
0,Land (This is only shown when no more specific...,https://wiki.openstreetmap.org/w/images/8/8a/B...,NaN
1,"Body of water (ocean, sea, pond, river) / swim...",https://wiki.openstreetmap.org/w/images/3/38/B...,NaN
2,Water body intermittent / Water body seasonal ...,https://wiki.openstreetmap.org/w/images/b/b7/W...,NaN
3,Reef,NaN,https://wiki.openstreetmap.org/w/images/4/41/N...
4,Natural woodland which is mostly or not at all...,https://wiki.openstreetmap.org/w/images/c/cc/R...,NaN


In [15]:
pd.read_csv("data/legend_area_clean.csv").head()

,Symbol,img
0,Land (This is only shown when no more specific...,https://wiki.openstreetmap.org/w/images/8/8a/B...
1,"Body of water (ocean, sea, pond, river) / swim...",https://wiki.openstreetmap.org/w/images/3/38/B...
2,Water body intermittent / Water body seasonal ...,https://wiki.openstreetmap.org/w/images/b/b7/W...
3,Reef,https://wiki.openstreetmap.org/w/images/4/41/N...
4,Natural woodland which is mostly or not at all...,https://wiki.openstreetmap.org/w/images/c/cc/R...


In [19]:
import os
import pandas as pd
import requests
from tqdm import tqdm
from urllib.parse import urlparse

BASE_OUTPUT = "./data"

FILES = {
    "legend_area_clean": "./data/legend_area_clean.csv",
    "legend_line_clean": "./data/legend_line_clean.csv",
    "legend_point_clean": "./data/legend_point_clean.csv",
}


def safe_filename(name: str) -> str:
    return "".join(c if c.isalnum() or c in ("-", "_") else "_" for c in str(name)).strip("_")


def get_extension(url: str) -> str:
    path = urlparse(url).path
    ext = os.path.splitext(path)[1]
    return ext if ext else ".png"


def download_image(url: str, save_path: str):
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        with open(save_path, "wb") as f:
            f.write(r.content)
    except Exception as e:
        print(f"[ERROR] Failed {url}: {e}")


def process_csv(csv_name, csv_path):
    df = pd.read_csv(csv_path)

    out_dir = os.path.join(BASE_OUTPUT, csv_name)
    os.makedirs(out_dir, exist_ok=True)

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Downloading {csv_name}"):
        symbol = safe_filename(row["Symbol"])
        url = row["img"]

        ext = get_extension(url)
        file_path = os.path.join(out_dir, f"{symbol}{ext}")

        download_image(url, file_path)


def main():
    for csv_name, csv_path in FILES.items():
        process_csv(csv_name, csv_path)


if __name__ == "__main__":
    main()

[ERROR] Failed https://wiki.openstreetmap.org/w/images/thumb/6/6f/Waste-basket-12.svg/12px-Waste-basket-12.svg.png: HTTPSConnectionPool(host='wiki.openstreetmap.org', port=443): Read timed out. (read timeout=20)
